In [ ]:
from typing_extensions import TypedDict, Literal
from typing import List
from langgraph.types import Command
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model("openai:gpt-4o-mini")

In [19]:
class State(TypedDict):
    document: str
    summary: str
    sentiment: str
    key_points: str
    recommendation: str
    final_analysis: str


In [20]:
def get_summary(state: State):
    response = llm.invoke(f"이 문서를 3문장으로 요약해줘: {state['document']}")
    return {
        "summary": response.content,
    }

def get_sentiment(state: State):
    response = llm.invoke(f"이 문서의 감정과 어조를 분석해줘: {state['document']}")
    return {
        "sentiment": response.content,
    }

def get_key_points(state: State):
    response = llm.invoke(f"이 문서의 가장 중요한 핵심 포인트 5가지를 정리해줘: {state['document']}")
    return {
        "key_points": response.content,
    }

def get_recommendation(state: State):
    response = llm.invoke(f"이 문서를 바탕으로 추천할 다음 단계 3가지를 제안해줘: {state['document']}")
    return {
        "recommendation": response.content,
    }

def get_final_analysis(state: State):
    response = llm.invoke(
        f"""
        다음 보고서를 종합적으로 분석해줘.

        문서 분석 보고서
        ========================

        요약:
        {state['summary']}
        
        감정 분석:
        {state['sentiment']}
        
        핵심 포인트:
        {state["key_points"]}
        
        추천 사항:
        {state.get('recommendation', "N/A")}
        """)
    return {
        "final_analysis": response.content,
    }

In [21]:
graph_builder = StateGraph(State)

graph_builder.add_node("get_summary", get_summary)
graph_builder.add_node("get_sentiment", get_sentiment)
graph_builder.add_node("get_key_points", get_key_points)
graph_builder.add_node("get_recommendation", get_recommendation)
graph_builder.add_node("get_final_analysis", get_final_analysis)

graph_builder.add_edge(START, "get_summary")
graph_builder.add_edge(START, "get_sentiment")
graph_builder.add_edge(START, "get_key_points")
graph_builder.add_edge(START, "get_recommendation")

graph_builder.add_edge("get_summary", "get_final_analysis")
graph_builder.add_edge("get_sentiment", "get_final_analysis")
graph_builder.add_edge("get_key_points", "get_final_analysis")
graph_builder.add_edge("get_recommendation", "get_final_analysis")
graph_builder.add_edge("get_final_analysis", END)

graph = graph_builder.compile()

#graph

In [23]:
with open("fed_transcript.md", "r", encoding="utf-8") as file:
    document = file.read()

for chunk in graph.stream(
    {"document": document},
    stream_mode="updates"
):
    print(chunk, "\n")


{'get_summary': {'summary': '연방공개시장위원회(FOMC)는 현재 경제 성장 둔화와 노동시장의 하방 위험 증가, 인플레이션의 상승이라는 상황에 대응하여 기준금리를 0.25%포인트 인하하고, 대차대조표 축소를 계속하기로 결정했습니다. 최근 경제 지표는 소비지출 증가세가 느려져 성장 둔화가 관찰되는 한편, 기업 설비투자와 무형자산 투자는 개선된 모습을 보이고 있으며, 주택 부문의 활동은 여전히 약한 상태를 유지하고 있습니다. 연준은 최대고용과 물가안정이라는 이중 책무에 따라 정책을 조정하며, 경제 상황과 변화하는 전망, 위험 균형을 바탕으로 향후 정책을 지속적으로 평가해 나갈 계획입니다.'}} 

{'get_key_points': {'key_points': '이 문서의 핵심 포인트 5가지를 정리하면 다음과 같습니다:\n\n1. **경제상황 점검 및 금리 인하**: 고용 시장의 하방 위험과 인플레이션의 상승세에 대응하기 위해, FOMC는 기준금리를 0.25%포인트 인하하고, 대차대조표 축소를 계속하기로 결정했습니다. 이는 최대고용과 물가안정이라는 연준의 이중 책무를 지원하기 위한 조치입니다.\n\n2. **경제 성장 둔화**: 최근 경제 지표는 경제활동 증가세가 완만해졌음을 시사하며, 특히 소비지출 증가세가 느려진 것이 원인으로 지적됩니다. 반면, 기업의 설비투자와 무형자산 투자는 개선된 모습을 보이고 있습니다.\n\n3. **노동시장 변화**: 실업률이 소폭 상승했으나 여전히 낮은 수준에 머물러 있으며, 비농업 고용 증가세가 둔화되었습니다. 이러한 고용 둔화는 이민 감소와 노동참가율 하락에 기인하며, 노동 수요와 공급이 동시에 둔화하는 이례적인 상황입니다.\n\n4. **인플레이션 동향**: 최근 인플레이션은 2022년 중반 이후 고점에서 낮아졌으나 여전히 2% 목표를 초과하고 있습니다. 특히, 상품 가격 상승이 인플레이션 상승의 주요 원인으로 지목되고 있으며, 서비스 부문에서는 디스인플레이션이 지속 중입니다.\n\n5. **정책적 시사점과 

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-CgDlydDCG9VXM3zFIyll7i3q on tokens per min (TPM): Limit 30000, Used 28855, Requested 1568. Please try again in 846ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}